<a href="https://colab.research.google.com/github/2004-tanu/Natural-Language-Processing/blob/main/NLP_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

                           Assignment No : 4
  
  Name : Lokhande Tanuja Bhaskar
  Roll No : 23107071
  Batch : A

Statement : LSTM for Part of Speech tagging: Implement LSTM-based model for Parts of Speech (POS)
tagging. Given a sequence of words, train the LSTM network to predict the corresponding POS tags
by learning word dependencies and context from a labeled dataset.

In [2]:
import pandas as pd

data = pd.read_csv("NER dataset.csv", encoding="latin1")

In [3]:
print("Head Data:\n", data.head())

Head Data:
     Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O


In [4]:
# 2. Fill missing sentence numbers
data["Sentence #"] = data["Sentence #"].fillna(method="ffill")

/tmp/ipython-input-240109512.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data["Sentence #"] = data["Sentence #"].fillna(method="ffill")


In [5]:
# 3. Create sentences and tags
sentences = data.groupby("Sentence #")["Word"].apply(list).values
tags = data.groupby("Sentence #")["POS"].apply(list).values

print("\nSample Sentences:\n", sentences[:3])



Sample Sentences:
 [list(['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.'])
 list(['Iranian', 'officials', 'say', 'they', 'expect', 'to', 'get', 'access', 'to', 'sealed', 'sensitive', 'parts', 'of', 'the', 'plant', 'Wednesday', ',', 'after', 'an', 'IAEA', 'surveillance', 'system', 'begins', 'functioning', '.'])
 list(['Helicopter', 'gunships', 'Saturday', 'pounded', 'militant', 'hideouts', 'in', 'the', 'Orakzai', 'tribal', 'region', ',', 'where', 'many', 'Taliban', 'militants', 'are', 'believed', 'to', 'have', 'fled', 'to', 'avoid', 'an', 'earlier', 'military', 'offensive', 'in', 'nearby', 'South', 'Waziristan', '.'])]


In [6]:
# 4. Tokenization
words = list(set(data["Word"]))
pos = list(set(data["POS"]))

word2idx = {w:i+1 for i,w in enumerate(words)}
tag2idx = {t:i+1 for i,t in enumerate(pos)}

X = [[word2idx[w] for w in s] for s in sentences]
y = [[tag2idx[t] for t in tag] for tag in tags]

print("\nTokenized Data:\n", X[:3])


Tokenized Data:
 [[12109, 33781, 16351, 13125, 4947, 16962, 27249, 19297, 33547, 14195, 3005, 22675, 28566, 17349, 33641, 14195, 14844, 33781, 5839, 11234, 27469, 9008, 11545, 24080], [6179, 18900, 29232, 30425, 31102, 19297, 8626, 31389, 19297, 9698, 31309, 3233, 33781, 14195, 23166, 4768, 14246, 16993, 6975, 13445, 11927, 19592, 11390, 7448, 24080], [28985, 16977, 30402, 25471, 7237, 13602, 22675, 14195, 4393, 9706, 19564, 14246, 23250, 34147, 11024, 29437, 1673, 18898, 19297, 13125, 13747, 19297, 31942, 6975, 20037, 13540, 12152, 22675, 15098, 22030, 32977, 24080]]


In [8]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# 5. Padding
max_len = max(len(s) for s in X)

X = pad_sequences(X, maxlen=max_len, padding="post")
y = pad_sequences(y, maxlen=max_len, padding="post")

y = [to_categorical(i, num_classes=len(tag2idx)+1) for i in y]
y = np.array(y)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed

# 6. LSTM Model
model = Sequential()
model.add(Embedding(len(word2idx)+1, 64))
model.add(LSTM(64, return_sequences=True))
model.add(TimeDistributed(Dense(len(tag2idx)+1, activation="softmax")))

model.compile(optimizer="adam", loss="categorical_crossentropy")

In [12]:
# 7. Train model
model.fit(X, y, epochs=3, batch_size=32)

Epoch 1/3
1499/1499 ━━━━━━━━━━━━━━━━━━━━ 187s 120ms/step - loss: 0.4761
Epoch 2/3
1499/1499 ━━━━━━━━━━━━━━━━━━━━ 202s 120ms/step - loss: 0.0228
Epoch 3/3
1499/1499 ━━━━━━━━━━━━━━━━━━━━ 200s 118ms/step - loss: 0.0130


In [13]:
# 8. Predict only one sentence
test_sentence = ["I", "live", "in", "India"]

In [14]:
test_seq = [word2idx.get(w, 0) for w in test_sentence]
test_seq = pad_sequences([test_seq], maxlen=max_len, padding="post")


In [15]:
pred = model.predict(test_seq)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [17]:
idx2tag = {v:k for k,v in tag2idx.items()}
idx2tag[0] = "-PAD-"  # Add mapping for padding token
result = [idx2tag[np.argmax(p)] for p in pred[0]]

In [18]:
print("\nTest Sentence:", test_sentence)
print("Predicted POS:", result[:len(test_sentence)])


Test Sentence: ['I', 'live', 'in', 'India']
Predicted POS: ['PRP', 'VBP', 'IN', 'NNP']


In [20]:
# Accuracy calculation
loss = model.evaluate(X, y)
print("\nModel Loss:", loss)
# To get accuracy from model.evaluate, the model needs to be recompiled with metrics=['accuracy']

1499/1499 ━━━━━━━━━━━━━━━━━━━━ 35s 23ms/step - loss: 0.0098

Model Loss: 0.00986703671514988


In [21]:
# Recompile the model to include accuracy as a metric
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=['accuracy'])

# Evaluate the model on the training data to get both loss and accuracy
loss, accuracy = model.evaluate(X, y)

print("\nModel Accuracy:", accuracy)

1499/1499 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - accuracy: 0.9967 - loss: 0.0098

Model Accuracy: 0.9966466426849365
